<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
LangSmith Evaluations Basics
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M15-LangSmith-Evaluations"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Übersicht
---

***Conditional Routing & Tool-Loop*** zeigte Conditional Routing und Tool-Loops – der Graph entscheidet dynamisch, welchen Pfad er nimmt.
**Dieses Modul** beantwortet eine grundlegende Frage: **Wie gut funktioniert mein Agent wirklich?**

LangSmith ist das **Observability- und Evaluation-Tool** von LangChain.
Es macht unsichtbare Agent-Entscheidungen sichtbar und messbar.




<p><font color='black' size="5">
Die vier Kernkonzepte
</font></p>

| Konzept | Beschreibung | Wann relevant? |
|---------|-------------|----------------|
| **Trace / Run** | Protokoll eines Agent-Aufrufs (Inputs, Outputs, Laufzeit, Kosten) | Immer – Development & Production |
| **Dataset** | Sammlung von Test-Fällen mit erwarteten Antworten | Vor Deployment, Regression-Tests |
| **Evaluation** | Automatisches Testen des Agents gegen ein Dataset | Qualitätssicherung |
| **Experiment** | Ein Evaluations-Run mit eindeutigem Namen | A/B-Testing, Modell-Vergleiche |



<p><font color='black' size="5">
Wann LangSmith einsetzen?
</font></p>


| Situation | Ohne LangSmith | Mit LangSmith |
|-----------|---------------|---------------|
| **Debugging** | ⚠️ print()-Debugging | ✅ Vollständige Trace-Sicht: Prompt → Response |
| **Qualitätssicherung** | ⚠️ Manuelle Tests | ✅ Dataset-basierte Evaluation mit Metriken |
| **Modell-Wechsel** | ⚠️ Bauchgefühl | ✅ A/B-Test: GPT-4o-mini vs. GPT-4o |
| **Production** | ❌ Kein Einblick | ✅ Latenz, Fehlerrate, Token-Kosten in Echtzeit |

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Ökosystem</font> </br></p>

diagram = '''
flowchart TB
    subgraph CODE["💻 Mein Agent-Code"]
        direction LR
        LC["LangChain / LangGraph"] --> LLM["LLM-Aufruf"]
        LLM --> TOOLS["Tool-Aufrufe"]
    end

    subgraph LS["📊 LangSmith"]
        direction TB
        TRACE[("Traces\n(jeder Run)")]
        DS[("Datasets\n(Test-Faelle)")]
        EXP[("Experimente\n(Evaluation-Runs)")]
        FB[("Feedback\n(Qualitaetsbewertung)")]
    end

    CODE -->|"automatisch\nge-trackt"| TRACE
    DS   -->|"fuehrt aus"| EXP
    EXP  -->|"erzeugt"| TRACE
    TRACE -->|"annotiert mit"| FB

    DEV["🧑‍💻 Entwickler\nDebugging"] -.->|"liest"| TRACE
    EVAL["📏 Evaluation\nRegression-Test"] -.->|"nutzt"| EXP

    style TRACE fill:#4CAF50,color:#fff
    style DS    fill:#2196F3,color:#fff
    style EXP   fill:#9C27B0,color:#fff
    style FB    fill:#FF9800,color:#fff
'''

mermaid(diagram, width=900)

# 2 | Tracing komplexer Workflows
---

LangSmith trackt LangChain-Operationen **automatisch** – sobald die Umgebungsvariablen gesetzt sind.
Jeder `graph.invoke()`-Aufruf erzeugt einen Trace mit allen Unter-Runs (Nodes, LLM-Calls, Tools).

**Trace-Hierarchie in LangGraph:**

```
Root Run: graph.invoke()          ← sichtbar in LangSmith als Haupt-Run
  ├── Node Run: "agent"           ← ein Eintrag pro Node-Aufruf
  │    └── LLM Run: gpt-4o-mini  ← Prompt + Response + Token-Kosten
  ├── Node Run: "tools"           ← ToolNode-Ausführung
  │    └── Tool Run: erklaere_konzept
  └── Node Run: "agent"           ← zweiter Durchlauf (Tool-Loop)
       └── LLM Run: gpt-4o-mini
```

**run_name, tags und metadata** machen Traces **filterbar und verständlich**:

```python
config = {
    "run_name": "Produktfrage-Agent",
    "tags":     ["m15", "qa", "gpt-4o-mini"],
    "metadata": {"user_id": "user-42", "version": "1.0"}
}
graph.invoke(input_data, config=config)
```

**Trace Previews anpassen** *(neu: Feb 2026, v0.13.10)*

Konfigurierbar, welche Inputs und Outputs in der Tracing-Tabelle angezeigt werden – besonders nützlich bei Custom Data Structures oder langen Texten:
- LangSmith Dashboard → Settings → **Trace Preview** → Felder auswählen
- Spart Zeit beim Durchsuchen vieler Traces im Projekt

**Kosten über den gesamten Agent-Stack** *(neu: Feb 2026)*

LangSmith zeigt Kosten jetzt für den **gesamten Workflow** – nicht nur LLM-Calls. Custom cost metadata kann für externe API-Calls, Tools oder andere Komponenten mitgeloggt werden:

```python
from langsmith import Client
client = Client()

# Kosten für externe API-Calls, Tools etc. manuell loggen
client.update_run(
    run_id=run_id,
    extra={"cost": {"total_cost": 0.005, "currency": "USD"}}
)
```

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from IPython.display import Image as IPImage

@tool
def erklaere_konzept(konzept: str) -> str:
    '''Erklaert ein KI-Konzept. Nutze dieses Tool fuer ALLE Anfragen zu KI-Begriffen.'''
    konzepte = {
        "langsmith":  "LangSmith ist das Observability- und Evaluation-Tool von LangChain. "
                      "Es trackt LLM-Aufrufe, ermoeglicht Dataset-Tests und A/B-Vergleiche.",
        "langgraph":  "LangGraph ist ein Framework fuer Multi-Agent-Systeme. "
                      "Es verwendet StateGraphen mit Nodes, Edges und Checkpointing.",
        "langchain":  "LangChain ist ein Framework fuer LLM-Anwendungen. "
                      "Es vereinheitlicht Modell-Zugriffe, Prompt-Templates und Chains.",
        "evaluation": "Evaluation misst die Qualitaet eines Agenten systematisch. "
                      "Dafuer werden Datasets mit Referenzantworten und Evaluatoren genutzt.",
        "tracing":    "Tracing protokolliert jeden LLM-Aufruf mit Inputs, Outputs und Metadaten. "
                      "LangSmith visualisiert diese Traces als Baum-Struktur.",
    }
    key = konzept.lower().strip()
    # Exakter Treffer
    if key in konzepte:
        return konzepte[key]
    # Partieller Treffer: erstes bekanntes Konzept im Suchbegriff
    for k, v in konzepte.items():
        if k in key:
            return v
    return f"Konzept '{konzept}' nicht in der Wissensbasis gefunden."

@tool
def vergleiche_tools(tool_a: str, tool_b: str) -> str:
    '''Vergleicht zwei KI-Tools oder Frameworks.'''
    vergleiche = {
        ("langchain", "langgraph"): "LangChain = Chains & Agents. LangGraph = State Machines.",
        ("langsmith", "langchain"): "LangChain = Ausfuehrung. LangSmith = Beobachtung & Evaluation.",
    }
    k1 = (tool_a.lower(), tool_b.lower())
    k2 = (tool_b.lower(), tool_a.lower())
    return vergleiche.get(k1, vergleiche.get(k2,
           f"Kein direkter Vergleich fuer {tool_a} vs {tool_b} verfuegbar."))

tools_liste = [erklaere_konzept, vergleiche_tools]
llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0).bind_tools(tools_liste)

class WissensState(TypedDict):
    messages: Annotated[list, add_messages]

def agent_node(state: WissensState) -> dict:
    system = {
        "role": "system",
        "content": (
            "Du bist ein KI-Kurs-Assistent. "
            "WICHTIG: Rufe IMMER das Tool 'erklaere_konzept' auf, wenn nach einem KI-Begriff "
            "gefragt wird (LangSmith, LangGraph, LangChain, Evaluation, Tracing usw.). "
            "Antworte niemals ohne Tool-Aufruf bei Konzept-Fragen. Antworte kompakt auf Deutsch."
        )
    }
    return {"messages": [llm.invoke([system] + state["messages"])]}

builder = StateGraph(WissensState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools_liste))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

wissens_graph = builder.compile()
print("✅ Wissens-Agent kompiliert")

In [ ]:
# Graph-Visualisierung: Struktur des Agenten
display(IPImage(wissens_graph.get_graph().draw_mermaid_png()))

In [ ]:
# Agent mit LangSmith-Konfiguration aufrufen
# Jeder Aufruf erzeugt automatisch einen Trace in LangSmith

anfragen = [
    "Was ist LangSmith und warum brauche ich es?",
    "Erklaere mir den Unterschied zwischen LangChain und LangGraph.",
    "Was bedeutet Tracing in KI-Systemen?",
]

for i, anfrage in enumerate(anfragen):
    cfg = {
        "run_name": f"Wissens-Anfrage-{i+1}",
        "tags":     ["m15", "tracing-demo", "wissens-agent"],
        "metadata": {"anfrage_nr": i + 1, "kategorie": "konzept-erklaerung"},
    }
    result = wissens_graph.invoke(
        {"messages": [HumanMessage(content=anfrage)]},
        config=cfg
    )
    antwort      = result["messages"][-1].content
    tool_aufrufe = sum(1 for m in result["messages"]
                       if hasattr(m, "tool_calls") and m.tool_calls)
    print(
        f'Anfrage {i+1}: {anfrage}  \n'
        f'Tool-Aufrufe: {tool_aufrufe}  \n'
        f'Antwort: {antwort}\n---'
    )

print("\n✅ Traces in LangSmith sichtbar → Projekt: M15-LangSmith-Evaluations")

In [ ]:
#@markdown   <p><font size="4" color='green'>  Trace-Hierarchie in LangSmith</font> </br></p>

diagram = '''
flowchart TD
    ROOT["📊 Root Run\ngraph.invoke()\nrun_name + tags + metadata"]

    subgraph TURN1["1. Agent-Schritt"]
        direction LR
        N1["Node Run\n'agent'"] --> LLM1["LLM Run\ngpt-4o-mini\nPrompt + Response"]
    end

    subgraph TURN2["2. Tool-Schritt"]
        direction LR
        N2["Node Run\n'tools'"] --> TOOL["Tool Run\nerklaere_konzept\nInput + Output"]
    end

    subgraph TURN3["3. Agent-Schritt (Final)"]
        direction LR
        N3["Node Run\n'agent'"] --> LLM2["LLM Run\ngpt-4o-mini\nFinale Antwort"]
    end

    ROOT --> TURN1 --> TURN2 --> TURN3

    style ROOT  fill:#4CAF50,color:#fff
    style LLM1  fill:#2196F3,color:#fff
    style TOOL  fill:#FF9800,color:#fff
    style LLM2  fill:#2196F3,color:#fff
'''

mermaid(diagram, width=700)

In [ ]:
from langsmith import traceable

# @traceable: Eigene Funktionen in den LangSmith-Trace einbetten
# Nuetzlich fuer Hilfsfunktionen, die NICHT Teil von LangChain sind

@traceable(name="Vorverarbeitung", tags=["m15", "preprocessing"])
def bereinige_anfrage(text: str) -> str:
    '''Bereinigt eine Nutzer-Anfrage: entfernt doppelte Leerzeichen und Whitespace.'''
    return " ".join(text.split()).strip()

@traceable(name="Vollstaendige-Pipeline", tags=["m15", "pipeline"])
def vollstaendige_pipeline(roher_text: str) -> str:
    '''Pipeline: Bereinigung → Agent → Antwort.'''
    sauberer_text = bereinige_anfrage(roher_text)
    result = wissens_graph.invoke(
        {"messages": [HumanMessage(content=sauberer_text)]},
        config={"run_name": "Agent-in-Pipeline", "tags": ["m15", "pipeline"]}
    )
    return result["messages"][-1].content

# Demo: Rohe Eingabe mit Tippfehlern und doppelten Leerzeichen
# Ziel: zeigen, dass die Pipeline Bereinigung + Agent-Aufruf kombiniert
antwort = vollstaendige_pipeline("  Was ist   LangSmith  genau?  ")
mprint(f'**Pipeline-Ergebnis:**\n{antwort}')
print("\n✅ Trace: Vollstaendige-Pipeline > Vorverarbeitung + Agent")

# 3 | Datasets erstellen
---

Ein **Dataset** ist eine Sammlung von Test-Fällen mit bekannten Eingaben und Referenzantworten.
Es ist die Grundlage für **reproduzierbare, messbare Evaluierung**.

**Warum Datasets?**

| Ohne Dataset | Mit Dataset |
|-------------|-------------|
| Jeder Test ist einmalig | Gleiche Tests bei jedem Run |
| Kein Vergleich möglich | Baseline: v1 vs. v2 vs. v3 |
| "Funktioniert bei mir" | Objektive Metriken |
| Manueller Aufwand | Automation mit `evaluate()` |

**Dataset-Struktur** – jedes Example hat `inputs` und `outputs`:

```python
{
    "inputs":  {"frage": "Was ist LangSmith?"},     # Eingabe an den Agent
    "outputs": {"antwort": "LangSmith ist..."}       # Erwartete Referenzantwort
}
```

**Best Practices**

| Aspekt | Empfehlung |
|--------|------------|
| **Größe** | 10–50 Examples für Entwicklung, 100+ für Production |
| **Qualität** | Manuell kuratiert – keine automatisch generierten Referenzen |
| **Diversität** | Edge Cases, Grenzfälle, typische Nutzerfragen |
| **Versionierung** | `dataset-v1`, `dataset-v2` für Vergleiche |

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  Dataset → Evaluation → Ergebnis</font> </br></p>

diagram = '''
flowchart LR
    subgraph CREATE["📝 Dataset erstellen"]
        direction TB
        D1["client.create_dataset()\nName + Beschreibung"]
        D2["client.create_example()\nInput + Referenz-Output"]
        D1 --> D2
    end

    subgraph EVAL["🧪 Evaluation"]
        direction TB
        E1["evaluate(agent_fn,\ndata=dataset,\nevaluators=[...])"]
        E2["Fuer jedes Example:\nagent_fn(inputs) → outputs"]
        E3["evaluator(inputs,\noutputs, reference) → score"]
        E1 --> E2 --> E3
    end

    subgraph RESULT["📊 Ergebnisse"]
        direction TB
        R1["Experiment in LangSmith\nmit Metriken pro Example"]
        R2["Aggregierte Scores\n(Accuracy, Latenz, Kosten)"]
        R1 --> R2
    end

    CREATE --> EVAL --> RESULT

    style CREATE fill:#2196F3,color:#fff
    style EVAL   fill:#9C27B0,color:#fff
    style RESULT fill:#4CAF50,color:#fff
'''

mermaid(diagram, width=900)

In [ ]:
from langsmith import Client

client = Client()

DATASET_NAME = "M15-KI-Konzepte-v1"

examples = [
    {
        "inputs":  {"frage": "Was ist LangSmith?"},
        "outputs": {"antwort": "LangSmith ist das Observability- und Evaluation-Tool von LangChain. "
                               "Es trackt LLM-Aufrufe und ermoeglicht Dataset-basierte Evaluierung."}
    },
    {
        "inputs":  {"frage": "Was ist LangGraph?"},
        "outputs": {"antwort": "LangGraph ist ein Framework fuer Multi-Agent-Systeme. "
                               "Es verwendet StateGraphen mit Nodes, Edges und Checkpointing."}
    },
    {
        "inputs":  {"frage": "Was ist Evaluation in KI-Systemen?"},
        "outputs": {"antwort": "Evaluation misst die Qualitaet eines Agenten systematisch. "
                               "Dafuer werden Datasets mit Referenzantworten und Evaluatoren genutzt."}
    },
    {
        "inputs":  {"frage": "Was ist Tracing?"},
        "outputs": {"antwort": "Tracing protokolliert jeden LLM-Aufruf mit Inputs, Outputs und Metadaten. "
                               "LangSmith visualisiert diese Traces als Baum-Struktur."}
    },
    {
        "inputs":  {"frage": "Was ist LangChain?"},
        "outputs": {"antwort": "LangChain ist ein Framework fuer LLM-Anwendungen. "
                               "Es vereinheitlicht Modell-Zugriffe, Prompt-Templates und Chains."}
    },
]

vorhandene = [ds.name for ds in client.list_datasets()]

if DATASET_NAME in vorhandene:
    dataset = next(ds for ds in client.list_datasets() if ds.name == DATASET_NAME)
    print(f"✅ Dataset wiedergefunden: '{DATASET_NAME}'")
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="KI-Konzept-Erklaerungen fuer M15-Evaluation"
    )
    for ex in examples:
        client.create_example(
            dataset_id=dataset.id,
            inputs=ex["inputs"],
            outputs=ex["outputs"]
        )
    print(f"✅ Dataset erstellt: '{DATASET_NAME}' mit {len(examples)} Examples")

alle_examples = list(client.list_examples(dataset_id=dataset.id))
print(f"\nExamples im Dataset: {len(alle_examples)}")
for ex in alle_examples:
    print(f"  • {ex.inputs.get('frage', '')[:60]}")

# 4 | Evaluations & Custom Evaluators
---

Die `evaluate()`-Funktion führt den Agenten **automatisch** über alle Dataset-Examples aus
und berechnet Metriken mit **Evaluatoren**.

**Zwei Evaluator-Typen**

| Typ | Methode | Stärke | Schwäche |
|-----|---------|--------|----------|
| **Custom Evaluator** | Python-Funktion, deterministische Logik | Schnell, kostenlos | Nur für exakte Metriken |
| **LLM-as-Judge** | LLM bewertet Antwortqualität | Flexibel, semantisch | Langsamer, Kosten |

**Evaluator-Signatur**

```python
def mein_evaluator(
    inputs:            dict,   # Aus dem Dataset: {"frage": ...}
    outputs:           dict,   # Agent-Ausgabe:   {"antwort": ...}
    reference_outputs: dict,   # Referenz:        {"antwort": ...}
) -> dict:
    score = ...   # float 0.0 - 1.0
    return {"key": "metrik-name", "score": score}
```

**evaluate()-Funktion**

```python
from langsmith.evaluation import evaluate

results = evaluate(
    mein_agent,                       # Callable: inputs → outputs
    data          = DATASET_NAME,     # Dataset-Name
    evaluators    = [mein_evaluator], # Liste von Evaluatoren
    experiment_prefix = "M15-v1",     # Präfix für Experiment-Name
    max_concurrency   = 1,            # Parallele Ausführungen
)
```

In [ ]:
from langsmith.evaluation import evaluate

# ─── Agent-Funktion fuer evaluate() ──────────────────────────────────────────
def wissens_agent_fn(inputs: dict) -> dict:
    '''Wrappt den Graph fuer evaluate().

    Eingabe:  inputs["frage"] aus dem Dataset
    Ausgabe:  outputs["antwort"] fuer den Evaluator
    '''
    result = wissens_graph.invoke(
        {"messages": [HumanMessage(content=inputs["frage"])]},
        config={"run_name": "Eval-Run", "tags": ["m15", "evaluation"]}
    )
    return {"antwort": result["messages"][-1].content}

# ─── Custom Evaluator: Schluesselwort-Overlap ─────────────────────────────────
def schluesselwort_evaluator(
    inputs:            dict,
    outputs:           dict,
    reference_outputs: dict,
) -> dict:
    '''Prueft, wie viele Schluesselbegriffe der Referenz in der Antwort auftauchen.'''
    antwort  = outputs.get("antwort", "").lower()
    referenz = reference_outputs.get("antwort", "").lower()
    ref_tokens = set(w for w in referenz.split() if len(w) >= 4)
    treffer    = sum(1 for w in ref_tokens if w in antwort)
    score      = treffer / len(ref_tokens) if ref_tokens else 0.0
    return {"key": "schluesselwort_overlap", "score": round(score, 3)}

# ─── Evaluation durchfuehren ──────────────────────────────────────────────────
print("Starte Evaluation...\n")

eval_results = evaluate(
    wissens_agent_fn,
    data              = DATASET_NAME,
    evaluators        = [schluesselwort_evaluator],
    experiment_prefix = "M15-Schluesselwort",
    max_concurrency   = 1,
)

print("\n✅ Evaluation abgeschlossen")
print(f"→ Experiment: {eval_results.experiment_name}")
print("→ Ergebnisse in LangSmith: Projekt M15-LangSmith-Evaluations")

# 5 | LLM-as-Judge
---

Der **LLM-as-Judge** nutzt ein Sprachmodell als Evaluator.
Das LLM bewertet die Antwortqualität **semantisch** – nicht nur nach exakten Treffern.

**Wann LLM-as-Judge?**

| Situation | Empfehlung |
|-----------|------------|
| Antwort muss semantisch korrekt sein (nicht exakt) | ✅ LLM-as-Judge |
| Kreative Antworten, die variieren dürfen | ✅ LLM-as-Judge |
| Einfache Fakten-Prüfung | ⚠️ Custom Evaluator reicht |
| Viele Examples / hohe Kosten | ⚠️ Sparsam einsetzen |

**Prompt-Design** – der Judge-Prompt definiert die Bewertungskriterien:

```python
judge_prompt = load_prompt(
    "https://github.com/ralf-42/Agenten/blob/main/05_prompt/m15_llm_judge_prompt.md",
    mode="T"  # system + human Template mit {frage}, {antwort}, {referenz}
)
```

> **Wichtig:** Präzise Kriterien und eine klare Skala (0.0–1.0) im Judge-Prompt.
> Ein vager Judge-Prompt erzeugt inkonsistente, nicht-reproduzierbare Bewertungen.

In [ ]:
# LLM-as-Judge Prompt laden
judge_prompt = load_prompt(
    "https://github.com/ralf-42/Agenten/blob/main/05_prompt/m15_llm_judge_prompt.md",
    mode="T")

judge_llm = init_chat_model("openai:o3")  # kein temperature (o3 unterstützt es nicht)

def llm_judge_evaluator(
    inputs:            dict,
    outputs:           dict,
    reference_outputs: dict,
) -> dict:
    '''LLM bewertet Antwortqualitaet auf Skala 0.0 - 1.0.'''
    prompt_value = judge_prompt.invoke({
        "frage":    inputs.get("frage", ""),
        "antwort":  outputs.get("antwort", ""),
        "referenz": reference_outputs.get("antwort", ""),
    })
    response = judge_llm.invoke(prompt_value.messages)
    try:
        score = float(response.content.strip())
        score = max(0.0, min(1.0, score))
    except ValueError:
        score = 0.5   # Fallback
    return {"key": "llm_qualitaet", "score": score}

# ─── Demo: Einzelner Judge-Aufruf ─────────────────────────────────────────────
test_inputs   = {"frage":   "Was ist LangSmith?"}
test_outputs  = {"antwort": "LangSmith ist ein Monitoring-Tool fuer LangChain-Anwendungen."}
test_referenz = {"antwort": "LangSmith ist das Observability- und Evaluation-Tool von LangChain."}

ergebnis = llm_judge_evaluator(test_inputs, test_outputs, test_referenz)
mprint(
    f'**LLM-Judge Demo**  \n'
    f'Frage: {test_inputs["frage"]}  \n'
    f'Antwort: {test_outputs["antwort"]}  \n'
    f'**Judge-Score:** `{ergebnis["score"]:.2f}` (Skala 0.0 - 1.0)'
)
print("✅ LLM-as-Judge Evaluator bereit")

In [ ]:
print("Starte LLM-as-Judge Evaluation (5 Examples)...\n")

judge_results = evaluate(
    wissens_agent_fn,
    data              = DATASET_NAME,
    evaluators        = [schluesselwort_evaluator, llm_judge_evaluator],
    experiment_prefix = "M15-LLM-Judge",
    max_concurrency   = 1,
)

print("\n✅ Evaluation mit LLM-as-Judge abgeschlossen")
print(f"→ Experiment: {judge_results.experiment_name}")
print("→ In LangSmith: Metriken 'schluesselwort_overlap' + 'llm_qualitaet' sichtbar")


<p><font color='black' size="5">
Ausblick: RAGAS – spezialisierte RAG-Evaluation
</font></p>



Fuer **RAG-Systeme** (*Agentic RAG*) gibt es mit [RAGAS](https://docs.ragas.io/) ein Framework mit RAG-spezifischen Metriken, die LangSmith Evaluations erganzen:

| Metrik | Frage |
|--------|-------|
| **Faithfulness** | Ist die Antwort durch die gefundenen Chunks belegbar? |
| **Answer Relevancy** | Beantwortet die Antwort tatsaechlich die gestellte Frage? |
| **Context Recall** | Wurden alle relevanten Chunks gefunden? |
| **Context Precision** | Wie viele gefundenen Chunks sind tatsaechlich relevant? |

```python
# pip install ragas
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy

results = evaluate(
    dataset=ragas_dataset,   # HuggingFace Dataset: question, contexts, answer, ground_truth
    metrics=[faithfulness, answer_relevancy],
)
```

**Wann RAGAS, wann LangSmith Evaluations?**

| | RAGAS | LangSmith Evaluations |
|---|---|---|
| **Fokus** | RAG-Pipeline (Retrieval + Generation) | Agenten-Verhalten allgemein |
| **Metriken** | Faithfulness, Recall, Precision | Custom + LLM-as-Judge |


> RAGAS-Ergebnisse koennen als Feedback an LangSmith gesendet werden – damit ist der gesamte RAG-Stack in einem Dashboard sichtbar.

# 6 | A/B-Testing
---

**A/B-Testing** vergleicht zwei Varianten eines Agenten systematisch auf demselben Dataset.
Der `experiment_prefix` identifiziert jede Variante in LangSmith.

**Typische Vergleiche**

| Variante A | Variante B | Frage |
|-----------|-----------|-------|
| `gpt-4o-mini` | `gpt-4o` | Lohnt sich das teurere Modell? |
| Kurzer System-Prompt | Detaillierter System-Prompt | Welcher Prompt ist besser? |
| Temperature 0.0 | Temperature 0.7 | Konsistenz vs. Kreativität? |

**Ablauf**

```python
# Variante A
results_a = evaluate(agent_v1, data=DATASET, experiment_prefix="M15-Variante-A")

# Variante B
results_b = evaluate(agent_v2, data=DATASET, experiment_prefix="M15-Variante-B")

# Vergleich: In LangSmith beide Experimente gleichzeitig auswählen
# → Side-by-Side-Vergleichstabelle
```

**Baseline fixieren** *(neu: Feb 2026)*

Seit Februar 2026 kann ein Experiment als **Baseline fixiert** werden. Alle nachfolgenden `evaluate()`-Runs werden automatisch dagegen verglichen – kein manuelles Auswählen mehr nötig:
- LangSmith Dashboard → Experiments → Run auswählen → **"Pin as Baseline"**
- Nachfolgende Experimente zeigen automatisch den Differenz-Score zur Baseline
- Ideal für kontinuierliches Verbessern: jede neue Version direkt gegen die fixierte Baseline vergleichen

**Pairwise Annotation Queues** *(neu: Dez 2025, v0.12.61)*

Für subjektive Vergleiche (Ton, Stil, Kreativität) bieten **Pairwise Annotation Queues** einen Side-by-Side-Vergleich zur manuellen Bewertung – ergänzend zum automatischen A/B-Test:
- LangSmith Dashboard → Annotation Queues → **Pairwise Queue** erstellen
- Zwei Antworten werden nebeneinander angezeigt → Annotator wählt die bessere
- Ideal wenn automatische Evaluatoren (Keywords, LLM-Judge) nicht ausreichen

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  A/B-Testing Workflow</font> </br></p>

diagram = '''
flowchart TD
    DS[("📊 Dataset\nM15-KI-Konzepte-v1")]

    subgraph A["🟢 Variante A: Kurzer Prompt"]
        direction TB
        A1["agent_v1(inputs)"] --> A2["Evaluatoren"] --> A3["Experiment A\nScores"]
    end

    subgraph B["🟡 Variante B: Detaillierter Prompt"]
        direction TB
        B1["agent_v2(inputs)"] --> B2["Evaluatoren"] --> B3["Experiment B\nScores"]
    end

    DS -->|"gleiche Examples"| A
    DS -->|"gleiche Examples"| B

    A3 --> COMPARE["🔍 LangSmith\nSide-by-Side-Vergleich"]
    B3 --> COMPARE

    COMPARE --> ENTSCH["✅ Entscheidung:\nWelche Variante gewinnt?"]

    style A       fill:#E8F5E9
    style B       fill:#FFF9C4
    style COMPARE fill:#9C27B0,color:#fff
    style ENTSCH  fill:#4CAF50,color:#fff
'''

mermaid(diagram, width=900)

In [ ]:
llm_basis = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

SYSTEM_KURZ   = "Du bist ein KI-Assistent. Nutze die Tools fuer Erklaerungen."
SYSTEM_DETAIL = (
    "Du bist ein erfahrener KI-Kurs-Assistent fuer LangChain-Technologien. "
    "Deine Aufgabe: Erklaere KI-Konzepte klar und praxisnah auf Deutsch. "
    "Nutze immer das Tool 'erklaere_konzept' fuer Definitionen. "
    "Beende deine Antwort mit einem konkreten Anwendungsbeispiel (1 Satz)."
)

def baue_graph(system_prompt: str):
    '''Baut einen Wissens-Agenten mit gegebenem System-Prompt.'''
    llm_v = llm_basis.bind_tools(tools_liste)

    def agent_fn(state: WissensState) -> dict:
        system = {"role": "system", "content": system_prompt}
        return {"messages": [llm_v.invoke([system] + state["messages"])]}

    b = StateGraph(WissensState)
    b.add_node("agent", agent_fn)
    b.add_node("tools", ToolNode(tools_liste))
    b.add_edge(START, "agent")
    b.add_conditional_edges("agent", tools_condition)
    b.add_edge("tools", "agent")
    return b.compile()

graph_v1 = baue_graph(SYSTEM_KURZ)
graph_v2 = baue_graph(SYSTEM_DETAIL)

print("✅ Variante A (kurzer Prompt) und B (detaillierter Prompt) bereit")
print(f"   Variante A System-Prompt: {len(SYSTEM_KURZ)} Zeichen")
print(f"   Variante B System-Prompt: {len(SYSTEM_DETAIL)} Zeichen")

In [ ]:
def agent_v1_fn(inputs: dict) -> dict:
    result = graph_v1.invoke(
        {"messages": [HumanMessage(content=inputs["frage"])]},
        config={"run_name": "Eval-Variante-A", "tags": ["m15", "variante-a"]}
    )
    return {"antwort": result["messages"][-1].content}

def agent_v2_fn(inputs: dict) -> dict:
    result = graph_v2.invoke(
        {"messages": [HumanMessage(content=inputs["frage"])]},
        config={"run_name": "Eval-Variante-B", "tags": ["m15", "variante-b"]}
    )
    return {"antwort": result["messages"][-1].content}

print("Starte A/B-Test...\n")

results_a = evaluate(
    agent_v1_fn,
    data              = DATASET_NAME,
    evaluators        = [llm_judge_evaluator],
    experiment_prefix = "M15-Variante-A-KurzPrompt",
    max_concurrency   = 1,
)
print("✅ Variante A abgeschlossen")

results_b = evaluate(
    agent_v2_fn,
    data              = DATASET_NAME,
    evaluators        = [llm_judge_evaluator],
    experiment_prefix = "M15-Variante-B-DetailPrompt",
    max_concurrency   = 1,
)
print("✅ Variante B abgeschlossen")
print("\n→ LangSmith: Beide Experimente unter 'M15-LangSmith-Evaluations' vergleichen")

In [ ]:
import statistics

def hole_antwortlaengen(agent_fn, anfragen_liste):
    '''Misst Antwortlaengen fuer eine Liste von Anfragen.'''
    return [len(agent_fn({"frage": f}).get("antwort", "")) for f in anfragen_liste]

test_anfragen = [
    "Was ist LangSmith?",
    "Was ist Tracing?",
    "Was ist Evaluation?",
]

laengen_a = hole_antwortlaengen(agent_v1_fn, test_anfragen)
laengen_b = hole_antwortlaengen(agent_v2_fn, test_anfragen)

mprint(
    '## A/B-Vergleich: Antwortlaengen\n\n'
    '| Variante | System-Prompt | Ø Zeichen |\n'
    '|----------|--------------|----------|\n'
    f'| **A** (kurz) | {len(SYSTEM_KURZ)} Zeichen | {statistics.mean(laengen_a):.0f} |\n'
    f'| **B** (detail) | {len(SYSTEM_DETAIL)} Zeichen | {statistics.mean(laengen_b):.0f} |\n\n'
    'Semantische Qualitaet → in LangSmith: `llm_qualitaet`-Score vergleichen.'
)

# 7 | Security-Basics
---

LangSmith-Tracing überträgt alle Inputs und Outputs an den LangSmith-Server.
Das bringt spezifische Security-Überlegungen.

| Risiko | Beschreibung | Gegenmaßnahme |
|--------|-------------|----------------|
| **PII-Leakage** | Namen, E-Mails in Traces sichtbar | PII vor dem Tracing filtern |
| **API-Key-Exposition** | Keys im Code oder Logs | Aus Umgebungsvariablen laden |
| **Dev/Prod-Vermischung** | Entwicklungs-Traces in Production | Separate Projekte |
| **Dataset-PII** | PII in Referenzantworten | Datasets manuell prüfen |

> **Faustregel:** Was im Trace landet, landet auf LangSmith-Servern.
> In Production: PII immer vor dem LLM-Aufruf anonymisieren.

In [ ]:
import re
from langchain_core.runnables import RunnableConfig

# ─── Muster 1: PII-Filter vor dem Tracing ─────────────────────────────────────
PII_PATTERNS = [
    (r'[\w\.-]+@[\w\.-]+\.\w+',             '[EMAIL]'),
    (r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b',        '[TELEFON]'),
    (r'\b[A-Z][a-z]+\s[A-Z][a-z]+\b',           '[NAME]'),
]

def filtere_pii(text: str) -> str:
    '''Ersetzt PII-Muster durch Platzhalter vor dem LLM-Aufruf.'''
    for muster, ersatz in PII_PATTERNS:
        text = re.sub(muster, ersatz, text)
    return text

# ─── Muster 2: Sichere config-Erstellung ──────────────────────────────────────
def erstelle_sichere_config(run_name: str, user_id: str, experiment: str = "") -> dict:
    '''Erstellt eine LangSmith-Config ohne PII in Metadata.'''
    return {
        "run_name": run_name,
        "tags":     ["m15", experiment] if experiment else ["m15"],
        "metadata": {
            "user_id":     user_id,       # ✅ Nur ID, keine E-Mail
            "environment": "dev",
        }
    }

# ─── Demo ─────────────────────────────────────────────────────────────────────
roher_input = "Mein Name ist Max Mustermann, E-Mail: max@example.com. Was ist LangSmith?"
gefiltert   = filtere_pii(roher_input)

mprint(
    f'**Roh-Input:**  `{roher_input[:65]}...`  \n'
    f'**Gefiltert:** `{gefiltert[:65]}...`'
)

# ─── Muster 3: Security-Checkliste ────────────────────────────────────────────
mprint(
    '## Security-Checkliste fuer LangSmith\n\n'
    '- ✅ `LANGSMITH_API_KEY` aus Umgebungsvariable (nie hardcoden)\n'
    '- ✅ Separate Projekte: `M15-dev`, `M15-staging`, `M15-prod`\n'
    '- ✅ PII vor LLM-Aufruf filtern (E-Mail, Name, Telefon)\n'
    '- ✅ Nur anonymisierte IDs in `metadata` (keine E-Mails)\n'
    '- ✅ Bei DSGVO-Relevanz: EU-Endpoint (`eu.api.smith.langchain.com`)\n'
    '- ✅ Tracing fuer hochsensible Operationen via `RunnableConfig(callbacks=[])` deaktivieren'
)

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M15-LangSmith-Evaluations", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.



<p><font color='black' size="5">
Evaluation eines eigenen Themen-Agenten
</font></p>

Bauen Sie einen Agenten zu einem selbst gewählten Thema und evaluieren Sie ihn systematisch.

**1. Agent erstellen**

Wählen Sie ein Thema (z.B. Rezepte, Reiseziele, Python-Bibliotheken, Sportregeln).
Definieren Sie **2 Tools** und einen passenden System-Prompt.

**2. Dataset erstellen**

5–10 Fragen mit Referenzantworten zu Ihrem Thema:

```python
examples = [
    {"inputs": {"frage": "..."}, "outputs": {"antwort": "..."}},
    # ...
]
```

**3. Evaluation mit beiden Methoden**

- **Custom Evaluator** (z.B. Keyword-Treffer, Antwortlänge > 50 Zeichen)
- **LLM-as-Judge** mit dem bereitgestellten Judge-Prompt (`m15_llm_judge_prompt.md`)

**4. A/B-Test**

Vergleichen Sie zwei Varianten (z.B. `temperature=0.0` vs. `temperature=0.7`
oder kurzer vs. detaillierter System-Prompt).

Welche Variante erzielt höhere Judge-Scores?

**Bonus:** Visualisieren Sie die Score-Verteilung mit `mermaid()` –
z.B. als `xychart-beta` Balkendiagramm oder als Tabelle im Markdown-Format.